Testing out functions in omeconvert.py

In [ ]:
import os
os.chdir('/Users/burkelawlor/Repos/hci-irae')

In [1]:
import tifffile as tf
import sys
import numpy as np
import cv2

In [3]:
def img_resize(img,scale_factor):
    width = int(np.floor(img.shape[1] * scale_factor))
    height = int(np.floor(img.shape[0] * scale_factor))
    return cv2.resize(img, (width, height), interpolation = cv2.INTER_AREA)

In [26]:
def write_ome_tif(filename, image, channel_names, photometric_interp, metadata, subresolutions):
    subresolutions = subresolutions

    fn = filename + ".ome.tif"

    with tf.TiffWriter(fn,  bigtiff=True, ) as tif:
        px_size_x = metadata['PhysicalSizeX']
        px_size_y = metadata['PhysicalSizeY']

        options = dict(
            photometric=photometric_interp,
            tile=(1024, 1024),
            maxworkers=4,
            compression='jpeg2000',
            compressionargs={'level':85},
            resolutionunit='CENTIMETER',
        )

        print("Writing pyramid level 0")
        tif.write(
            image,
            subifds=subresolutions,
            resolution=(1e4 / px_size_x, 1e4 / px_size_y),
            metadata=metadata,
            **options
        )

        scale = 1
        for i in range(subresolutions):
            scale /= 2
            if photometric_interp == 'minisblack':
                if image.shape[0] < image.shape[-1]:
                    image = np.moveaxis(image,0,-1)
                    image = img_resize(image,0.5)
                    image = np.moveaxis(image,-1,0)
            else:
                image = img_resize(image,0.5)

            print("Writing pyramid level {}".format(i+1))
            tif.write(
                image,
                subfiletype=1,
                resolution=(1e4 / scale / px_size_x, 1e4 / scale / px_size_y),
                **options
            )


In [43]:
input = "26697x1_0076657_region_1.ome.tiff"

tif = tf.TiffFile(input)
image = tif.asarray()

channel_names=None
print(tif.pages[0].samplesperpixel)
print(tif.pages[0].get_resolution())
print(tif.pages[0].resolutionunit)


3
(0, 0)
1


In [48]:
meta = tf.xml2dict(tif.ome_metadata)
meta

{'OME': {'Instrument': {'Microscope': {'Model': 'Axioscan 7',
    'Type': 'Upright'},
   'Filament': {'ID': 'LightSource:1', 'Type': 'Halogen'},
   'Detector': {'Model': 'Axiocam705c', 'ID': 'Detector:1'},
   'Objective': {'Model': 'Plan-Apochromat 20x/0.8 M27',
    'ID': 'Objective:1',
    'Immersion': 'Air',
    'LensNA': 0.8,
    'NominalMagnification': 20,
    'WorkingDistance': 610},
   'ID': 'Instrument:1'},
  'Image': {'AcquisitionDate': '2025-10-22T21:18:32.0744423Z',
   'Description': None,
   'InstrumentRef': {'ID': 'Instrument:1'},
   'ObjectiveSettings': {'ID': 'Objective:1',
    'Medium': 'Air',
    'RefractiveIndex': 1.000293},
   'Pixels': {'Channel': {'DetectorSettings': {'ID': 'Detector:1',
      'Binning': '1x1'},
     'ID': 'Channel:1:0',
     'Name': 'Bright',
     'SamplesPerPixel': 3,
     'IlluminationType': 'Transmitted',
     'AcquisitionMode': 'WideField',
     'ContrastMethod': 'Brightfield',
     'Fluor': 'Bright'},
    'TiffData': None,
    'Plane': {'TheZ'

In [33]:
tif.pages[0].get_resolution()

(0, 0)

In [31]:
write_ome_tif(filename, image, channel_names, photometric_interp, metadata, subresolutions = 7)

Writing pyramid level 0
Writing pyramid level 1
Writing pyramid level 2
Writing pyramid level 3
Writing pyramid level 4
Writing pyramid level 5
Writing pyramid level 6
Writing pyramid level 7


In [ ]:
input = "26697x1_0076657_region_1.ome.tiff"
tif = tf.TiffFile(filename)
image = tif.asarray()

channel_names=None

if tif.pages[0].samplesperpixel == 3:
    photometric_interp='rgb'
elif tif.pages[0].samplesperpixel == 1:
    photometric_interp='minisblack'

if tif.pages[0].get_resolution():
    res = tif.pages[0].get_resolution()
    if tif.pages[0].resolutionunit == 2:
        unit = "in"
    elif tif.pages[0].resolutionunit == 3:
        unit = "cm"
    elif tif.pages[0].resolutionunit == 4:
        unit = "mm"
    elif tif.pages[0].resolutionunit == 5:
        unit = "um"

if tif.is_ome:
    meta = tf.xml2dict(tif.ome_metadata)
    res = (meta['OME']['Image']['Pixels']['PhysicalSizeX'],meta['OME']['Image']['Pixels']['PhysicalSizeY'])
    # unit = meta['OME']['Image']['Pixels']['PhysicalSizeXUnit']

    try:
        channel_names=[]
        for i, element in enumerate(meta['OME']['Image']['Pixels']['Channel']):
            channel_names.append(meta['OME']['Image']['Pixels']['Channel'][i]['Name'])
    except KeyError:
        channel_names=None

metadata={
    'PhysicalSizeX': res[0],
    'PhysicalSizeXUnit': unit,
    'PhysicalSizeY': res[1],
    'PhysicalSizeYUnit': unit,
    'Channel': {'Name': channel_names}
    }


In [19]:
if tif.pages[0].get_resolution():
    res = tif.pages[0].get_resolution()
    if tif.pages[0].resolutionunit == 2:
        unit = "in"
    elif tif.pages[0].resolutionunit == 3:
        unit = "cm"
    elif tif.pages[0].resolutionunit == 4:
        unit = "mm"
    elif tif.pages[0].resolutionunit == 5:
        unit = "um"

unit

'cm'

In [20]:
tif.pages[0].samplesperpixel

3

In [12]:
filename = "converted.ome.tif"
tif = tf.TiffFile(filename)
image = tif.asarray()

if tif.is_ome:
    print('is ome')
    meta = tf.xml2dict(tif.ome_metadata)

meta['OME']['Image']['Pixels']

is ome


{'Channel': {'LightPath': None, 'ID': 'Channel:0', 'SamplesPerPixel': 3},
 'TiffData': {'UUID': {'FileName': 'converted.ome.tif',
   'value': 'urn:uuid:ed499475-8b3d-420a-a6ce-4b0f1c4b32b0'},
  'FirstC': 0,
  'FirstT': 0,
  'FirstZ': 0,
  'IFD': 0,
  'PlaneCount': 1},
 'BigEndian': True,
 'DimensionOrder': 'XYCZT',
 'ID': 'Pixels:0',
 'Interleaved': True,
 'PhysicalSizeX': 0.34402847,
 'PhysicalSizeXUnit': 'µm',
 'PhysicalSizeY': 0.34402847,
 'PhysicalSizeYUnit': 'µm',
 'SizeC': 3,
 'SizeT': 1,
 'SizeX': 5670,
 'SizeY': 14548,
 'SizeZ': 1,
 'Type': 'uint8'}